# Gold Layer – Customer Dimension

This notebook builds the Customer Dimension table in the Gold layer by integrating customer master data from CRM with demographic and location information from ERP systems. The resulting dataset provides a unified customer view for reporting and analytical purposes.

In [0]:
df = spark.sql("""
SELECT 
    ROW_NUMBER() OVER(ORDER BY ci.customer_id ) AS  customer_key, 
    ci.customer_id ,
    ci.customer_number , 
    ci.first_name ,
    ci.last_name , 
    cl.country ,  
    CASE 
        WHEN ci.gender  IN('Male','Female') THEN ci.gender 
        WHEN cb.gender IN('Male' , 'Female') THEN  cb.gender 
        ELSE 'N/A'
    END AS gender  , 
    ci.marital_status , 
    cb.birth_date , 
    ci.created_date 

FROM workspace.silver.crm_cust_info ci
LEFT JOIN workspace.silver.erp_customer_location cl
ON ci.customer_number = cl.customer_number
LEFT JOIN workspace.silver.erp_customers cb 
ON ci.customer_number = cb.customer_number 
"""
)


## Preview Dimension Data

Review the customer dimension before loading it into the Gold layer.

In [0]:
df.limit(10).display()

## Load to Gold Layer

Write the customer dimension table to the Gold layer.

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.gold.dim_customers')

## Verify Data Load

Validate that the customer dimension has been successfully loaded into the Gold layer.

In [0]:
%sql
SELECT * 
FROM workspace.gold.dim_customers LIMIT 10